# Stickman Motion Diffusion Training (GPU)

This notebook trains the **Temporal Transformer Motion Denoiser** on a GPU (Kaggle T4/P100 or Google Colab T4) following `AGENTS.md` specifications:
- 1 token per frame, up to 120 frames (5s @ 24fps)
- Hidden dimension 256, 6 transformer blocks (~10M parameters)
- Predicts clean motion $x_0$
- Compound Recon + Velocity loss with dynamic padding masks
- Automated export to ONNX (`models/motion_denoiser.onnx`) for dev laptop CPU inference

In [ ]:
# 1. Environment & GPU Check
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. For fast training, enable GPU runtime (T4/P100).")

In [ ]:
# 2. Install dependencies if running in Colab/Kaggle
!pip install -q onnx onnxruntime pillow numpy
!apt-get -y install ffmpeg > /dev/null 2>&1

In [ ]:
# 3. Verify Project Imports
import os
import sys
sys.path.append(".")

from src.model import MotionDenoiser
from src.data_gen import gen_single, ACTIONS_1P
from src.train import train_motion_model

print(f"Loaded {len(ACTIONS_1P)} 1P action classes: {sorted(list(ACTIONS_1P))}")

In [ ]:
# 4. Run Diffusion Training Pipeline
# Synthesizes 5,000 clips and trains for 50 epochs
model = train_motion_model(
    epochs=50,
    batch_size=64,
    lr=1e-4,
    num_samples=5000,
    timesteps=1000,
    output_onnx="models/motion_denoiser.onnx",
    checkpoint_dir="checkpoints"
)

In [ ]:
# 5. Verify Exported ONNX Model on CPU
from src.onnx_runner import MotionONNXRunner
runner = MotionONNXRunner("models/motion_denoiser.onnx")
print(f"ONNX Model Loaded: {runner.is_model_loaded()}")

if runner.is_model_loaded():
    sample = runner.generate_motion("walk", duration_s=3.0, speed=1.0)
    print(f"Generated motion shape: {sample['feat'].shape} (T={sample['T']} frames)")

In [ ]:
# 6. Render Sample Motion Video
from src.renderer import render_to_video

if runner.is_model_loaded():
    sample = runner.generate_motion("walk", duration_s=3.0, speed=1.0)
    render_to_video(sample["feat"], "out/test_diffusion_walk.mp4", n_person=1)
    print("Rendered out/test_diffusion_walk.mp4!")